# Hotel Review Classification — Training (Kaggle)

Trains two separate **pretrained `bert-base-uncased`** models on the HRAST
hotel-review dataset (fine-tuning), in the style of NLP Lab 5 (an
encoder-only Transformer + pooling + a linear classifier):

1. A **sentiment model**: one label per sentence (`negative`, `neutral`, `positive`).
2. An **aspect model**: which of 21 hotel aspects (Staff, Breakfast, Wi-Fi, ...)
   a sentence mentions. One sentence can mention several aspects.

## What we write by hand, and what we take from a library

| Part | How it is done |
|---|---|
| Text cleaning, duplicate removal | by hand (plain Python / pandas) |
| Padding and attention mask | **by hand** |
| Batching and shuffling | **by hand** |
| Classification head (dropout + linear layer) | **by hand** |
| Training loop (forward, loss, backward, step) | **by hand** |
| Accuracy, precision, recall, F1, confusion matrices | **by hand** |
| Threshold search for the aspect model | **by hand** |
| Diagrams | `matplotlib` (we compute every number ourselves, the library only draws) |
| Pretrained BERT encoder (embeddings, positional encoding, self-attention layers) | library: `AutoModel` |
| WordPiece tokenizer that matches BERT's vocabulary | library: `AutoTokenizer` (text -> IDs only) |
| Saving in the format the web app loads | library: `save_pretrained` |
| Train / validation / test split that keeps rare labels | library: `iterative-stratification` |

The BERT encoder is the same idea as Lab 5's `TransformerEncoder`, but it is
already **pretrained** on a huge amount of text. We keep it and only add our
own small classifier on top, then fine-tune everything on the hotel reviews.

## Diagrams this notebook produces

Every diagram is shown in the notebook and saved as a PNG in `/kaggle/working/plots`.

- Data: sentiment distribution, aspect frequency, review length histogram
- Sentiment model: training curves, **confusion matrix**, per-class precision / recall / F1
- Aspect model: training curves, threshold search curve, per-aspect precision / recall / F1,
  **one confusion matrix per aspect**

## Before you run this

1. **Turn on a GPU.** Notebook settings (right panel) -> Accelerator -> GPU.
2. **Turn on Internet.** Needed to download `bert-base-uncased` and install
   one small package.
3. **Attach the dataset.** Add Data -> upload `dataset/HRAST.csv` as a Kaggle
   Dataset and attach it. The notebook finds it under `/kaggle/input/`.

## After it finishes: using the models in the web app

Both models are saved in the same Hugging Face format the project's web app
(`src/web/app.py`) loads, in these folders:

- `/kaggle/working/artifacts/sentiment`
- `/kaggle/working/artifacts/aspects` (also contains `thresholds.json`)

The last cell also packs them into **`hotel_models.zip`**. Click **Save Version ->
Save & Run All (Commit)**, open the version's **Output** tab, download
`hotel_models.zip`, and unzip it in the project root so you get
`artifacts/sentiment` and `artifacts/aspects`. Then run `python src/web/app.py`.

## Step 0: Install, import, and set up

In [ ]:
# Kaggle already has PyTorch and Transformers installed.
# We only need this one extra package, for the stratified train/val/test split.
!pip install --quiet iterative-stratification==0.1.9

In [ ]:
import re
import json
import glob
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [ ]:
# Use the GPU if Kaggle gave us one, otherwise fall back to CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("No GPU found. Training will be slow on CPU.")

# Fixed seed so the split and training are reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Kaggle keeps everything written under /kaggle/working as this notebook's output.
WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
ARTIFACTS_DIR = WORKING_DIR / "artifacts"
SENTIMENT_DIR = ARTIFACTS_DIR / "sentiment"
ASPECT_DIR = ARTIFACTS_DIR / "aspects"
PLOTS_DIR = WORKING_DIR / "plots"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
ASPECT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Find the HRAST.csv file from the dataset you attached under /kaggle/input.
csv_files = glob.glob("/kaggle/input/**/HRAST.csv", recursive=True)
if len(csv_files) == 0:
    raise FileNotFoundError("HRAST.csv not found under /kaggle/input. Did you attach the dataset?")
RAW_CSV_PATH = csv_files[0]
print("Found dataset at:", RAW_CSV_PATH)

## Step 1: Define the labels

Sentiment is one label per sentence. Aspects are 21 yes/no labels per sentence.

In [ ]:
SENTIMENT_COLUMNS = ["positive", "negative", "neutral"]
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SENTIMENT_TO_ID = {"negative": 0, "neutral": 1, "positive": 2}

ASPECT_COLUMNS = [
    "Clean", "Comfort", "Facilities/Amenities", "Location",
    "Restaurant (dinner)", "Staff", "View (Balcony)", "Breakfast", "Room",
    "Pool", "Beach", "Bathroom/Shower (toilet)", "Bar", "Bed", "Parking",
    "Noise", "Reception-checkin", "Lift", "Value for money", "Wi-Fi", "Generic",
]

LABEL_COLUMNS = SENTIMENT_COLUMNS + ASPECT_COLUMNS

## Step 2: Load and clean the data

We go through the raw file one simple check at a time: drop the empty
column, tidy up the text, remove rows with bad labels, and remove duplicate
reviews that disagree on their labels.

In [ ]:
df = pd.read_csv(RAW_CSV_PATH, encoding="utf-8-sig")
print("Rows in raw file:", len(df))

# The raw file has one empty, unnamed column. Drop it.
for column in df.columns:
    if str(column).startswith("Unnamed"):
        df = df.drop(columns=[column])

In [ ]:
def clean_text(text):
    # Collapse repeated spaces/tabs/newlines into a single space.
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["review"] = df["review"].apply(clean_text)
df = df[df["review"] != ""]
print("Rows after removing empty reviews:", len(df))

In [ ]:
# Every sentiment/aspect cell must be exactly 0 or 1. Drop rows where it is not.
def row_labels_are_binary(row):
    for column in LABEL_COLUMNS:
        if row[column] not in (0, 1, "0", "1"):
            return False
    return True

keep_mask = df.apply(row_labels_are_binary, axis=1)
df = df[keep_mask]
print("Rows after removing non-binary label values:", len(df))

In [ ]:
# Every row must have exactly one sentiment label.
for column in SENTIMENT_COLUMNS:
    df[column] = df[column].astype(int)
for column in ASPECT_COLUMNS:
    df[column] = df[column].astype(int)

sentiment_total = df["positive"] + df["negative"] + df["neutral"]
df = df[sentiment_total == 1]
print("Rows after keeping exactly one sentiment label:", len(df))

In [ ]:
# Turn the three sentiment columns into one readable label.
def sentiment_name(row):
    if row["negative"] == 1:
        return "negative"
    if row["neutral"] == 1:
        return "neutral"
    return "positive"

df["sentiment"] = df.apply(sentiment_name, axis=1)

In [ ]:
# If the same review text shows up more than once, only keep it when every
# copy has the exact same labels. If copies disagree, we cannot trust which
# one is right, so we drop all of them.
rows_to_keep = []

for review_text, group in df.groupby("review"):
    first_row_labels = group[LABEL_COLUMNS].iloc[0].tolist()
    all_copies_agree = True
    for row_number in range(len(group)):
        this_row_labels = group[LABEL_COLUMNS].iloc[row_number].tolist()
        if this_row_labels != first_row_labels:
            all_copies_agree = False
    if all_copies_agree:
        rows_to_keep.append(group.iloc[0])

df = pd.DataFrame(rows_to_keep).reset_index(drop=True)
print("Rows after removing duplicate reviews:", len(df))

In [ ]:
print("Final cleaned rows:", len(df))
# This exact number is documented in the project plan for this dataset.
assert len(df) == 23095, f"Expected 23095 clean rows, got {len(df)}"

## Step 3: Split into train / validation / test

We use a 70/15/15 split. `MultilabelStratifiedShuffleSplit` keeps rare
aspects and the small neutral-sentiment class represented in every split.

In [ ]:
stratify_columns = ASPECT_COLUMNS + SENTIMENT_COLUMNS
label_matrix = df[stratify_columns].to_numpy()

# First split off 70% for training, 30% left over.
splitter_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_index, rest_index = next(splitter_1.split(df, label_matrix))

train_df = df.iloc[train_index].reset_index(drop=True)
rest_df = df.iloc[rest_index].reset_index(drop=True)

# Split the remaining 30% evenly into validation (15%) and test (15%).
rest_label_matrix = rest_df[stratify_columns].to_numpy()
splitter_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
validation_index, test_index = next(splitter_2.split(rest_df, rest_label_matrix))

validation_df = rest_df.iloc[validation_index].reset_index(drop=True)
test_df = rest_df.iloc[test_index].reset_index(drop=True)

print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

train_df.to_csv(DATA_DIR / "train.csv", index=False)
validation_df.to_csv(DATA_DIR / "validation.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)

### Diagram: what does the data look like?

Two bar charts of the cleaned data. Look at how few `neutral` sentences there
are and how rare some aspects (Lift, Beach, Parking) are. This is why we use
macro-F1 (every class counts equally) instead of only accuracy.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 2]})

# Left: how many sentences per sentiment.
sentiment_counts = [int((df["sentiment"] == name).sum()) for name in SENTIMENT_LABELS]
ax1.bar(SENTIMENT_LABELS, sentiment_counts, color=["tab:red", "tab:gray", "tab:green"])
for i, count in enumerate(sentiment_counts):
    ax1.text(i, count, str(count), ha="center", va="bottom")
ax1.set_title("Sentiment distribution")
ax1.set_ylabel("Number of sentences")

# Right: how many sentences mention each aspect (sorted).
aspect_counts_all = [int(df[column].sum()) for column in ASPECT_COLUMNS]
order = sorted(range(len(ASPECT_COLUMNS)), key=lambda i: aspect_counts_all[i])
ax2.barh([ASPECT_COLUMNS[i] for i in order], [aspect_counts_all[i] for i in order], color="tab:blue")
ax2.set_title("Aspect frequency (a sentence can have several aspects)")
ax2.set_xlabel("Number of sentences")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "data_distribution.png", dpi=150)
plt.show()

## Step 4: Training settings

Same settings for both models.

In [ ]:
BASE_MODEL = "bert-base-uncased"
MAX_LENGTH = 128        # longer reviews are cut to 128 tokens
EPOCHS = 5
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
DROPOUT = 0.1

## Step 5: Tokenizer (text -> numbers)

A neural network only understands numbers, so every review becomes a list of
token IDs. BERT was pretrained with one fixed WordPiece vocabulary, so we
must use its tokenizer (a different vocabulary would make the pretrained
weights meaningless). We only use it to turn text into IDs. Padding and
masks are done by hand below.

BERT adds two special tokens to every review: `[CLS]` at the start and
`[SEP]` at the end. The filler token is `[PAD]` (ID 0).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
PAD_ID = tokenizer.pad_token_id

example = "The staff were friendly, and the Wi-Fi was free!"
print(tokenizer.tokenize(example))
print(tokenizer(example)["input_ids"])

In [ ]:
train_texts = train_df["review"].tolist()
validation_texts = validation_df["review"].tolist()
test_texts = test_df["review"].tolist()


def encode_all(texts):
    # Texts -> one list of token IDs per review (cut to MAX_LENGTH, no padding yet).
    return tokenizer(texts, truncation=True, max_length=MAX_LENGTH)["input_ids"]


# Encode every review once, so we do not repeat this work every epoch.
train_ids = encode_all(train_texts)
validation_ids = encode_all(validation_texts)
test_ids = encode_all(test_texts)

print("Example text:     ", train_texts[0])
print("Example token IDs:", train_ids[0])

### Diagram: how long are the reviews?

We cut reviews to `MAX_LENGTH` tokens. This histogram checks that almost no
review is actually cut.

In [ ]:
lengths = [len(ids) for ids in train_ids]
too_long = sum(1 for length in lengths if length >= MAX_LENGTH)
print(f"Longest review: {max(lengths)} tokens, average: {sum(lengths) / len(lengths):.1f} tokens")
print(f"Reviews that reach the {MAX_LENGTH}-token limit: {too_long} of {len(lengths)}")

plt.figure(figsize=(7, 4))
plt.hist(lengths, bins=40, color="tab:blue")
plt.axvline(MAX_LENGTH, color="red", linestyle="--", label=f"MAX_LENGTH = {MAX_LENGTH}")
plt.title("Review length in tokens (training set)")
plt.xlabel("Tokens per review")
plt.ylabel("Number of reviews")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "review_lengths.png", dpi=150)
plt.show()

### Batching and padding helpers (used by both models)

- `make_batches` cuts the data into fixed-size batches, in random order for
  training and in the original order for evaluation.
- `pad_batch` pads every review in a batch to the length of the longest one and
  builds the **attention mask** (1 for a real token, 0 for `[PAD]`), so the
  attention layers ignore the filler tokens.

In [ ]:
def make_batches(ids_list, labels, batch_size, shuffle):
    indices = list(range(len(ids_list)))
    if shuffle:
        random.shuffle(indices)

    batches = []
    start = 0
    while start < len(indices):
        end = start + batch_size
        batch_indices = indices[start:end]
        batch_ids = [ids_list[i] for i in batch_indices]
        batch_labels = [labels[i] for i in batch_indices]
        batches.append((batch_ids, batch_labels))
        start = end

    return batches


def pad_batch(batch_ids):
    longest = max(len(ids) for ids in batch_ids)
    longest = max(longest, 1)   # avoid a zero-length batch if a review is empty

    padded = []
    for ids in batch_ids:
        padded.append(ids + [PAD_ID] * (longest - len(ids)))

    inputs = torch.tensor(padded, dtype=torch.long).to(device)
    attention_mask = (inputs != PAD_ID).long()     # 1 = real token, 0 = [PAD]
    return inputs, attention_mask

## Step 6: The classifier model

For each review:

1. **Pretrained BERT encoder** (library): embeddings + positional information +
   stacked self-attention layers turn the token IDs into vectors.
2. **Pooled vector**: BERT's summary vector for the whole review (built from
   the `[CLS]` token).
3. **Dropout** (our class): randomly zeroes some values while training so the
   model does not over-memorise.
4. **Linear layer** (our own classification head): turns the vector into one
   score (a *logit*) per class.

The same class is used for both models. Only the number of output classes
differs: 3 for sentiment, 21 for aspects.

In [ ]:
class BertClassifier(nn.Module):
    def __init__(self, num_classes, dropout):
        super().__init__()
        self.bert = AutoModel.from_pretrained(BASE_MODEL)      # pretrained weights
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

        # Start the new head with small random weights (same as BERT's own init).
        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask):
        # 1. Run the pretrained encoder. The attention mask hides [PAD] tokens.
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # 2. One summary vector per review, shape (batch, 768).
        pooled = output.pooler_output

        # 3. Dropout, then one score (logit) per class.
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits


def build_model(num_classes):
    model = BertClassifier(num_classes, DROPOUT)
    return model.to(device)

### Shared training and prediction functions

Both models use the same loop; the only differences are the loss function and
how the label tensor is made. `get_logits` runs the model on a whole dataset
without training and returns one row of logits per review.

In [ ]:
def train_one_epoch(model, optimizer, loss_function, ids_list, labels, label_dtype):
    model.train()
    batches = make_batches(ids_list, labels, TRAIN_BATCH_SIZE, shuffle=True)
    total_loss = 0.0

    for batch_ids, batch_labels in batches:
        inputs, attention_mask = pad_batch(batch_ids)
        labels_tensor = torch.tensor(batch_labels, dtype=label_dtype).to(device)

        optimizer.zero_grad()
        logits = model(inputs, attention_mask)
        loss = loss_function(logits, labels_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(batches)
    return average_loss


def get_logits(model, ids_list):
    model.eval()
    dummy_labels = [0] * len(ids_list)
    batches = make_batches(ids_list, dummy_labels, EVAL_BATCH_SIZE, shuffle=False)

    all_logits = []
    with torch.no_grad():
        for batch_ids, _ in batches:
            inputs, attention_mask = pad_batch(batch_ids)
            logits = model(inputs, attention_mask)
            all_logits.append(logits.cpu())

    return torch.cat(all_logits, dim=0)

### Metrics written by hand

For one class, with `tp` = true positives, `fp` = false positives and
`fn` = false negatives:

- **Precision** = `tp / (tp + fp)`: of everything we predicted as this class, how much was right.
- **Recall** = `tp / (tp + fn)`: of everything that really is this class, how much we found.
- **F1** = `2*tp / (2*tp + fp + fn)`: one number that balances precision and recall.
  (Each is defined as 0 if its bottom is 0.)

A **confusion matrix** counts, for every true class (rows), how often each
class was predicted (columns). The diagonal is the correct predictions.

- **Macro-F1**: compute F1 for every class, then take the plain average.
- **Micro-F1**: add up `tp`, `fp`, `fn` over all classes first, then compute one F1.

In [ ]:
def f1_from_counts(tp, fp, fn):
    bottom = 2 * tp + fp + fn
    if bottom == 0:
        return 0.0
    return 2 * tp / bottom


def precision_recall_f1(tp, fp, fn):
    precision = 0.0
    if tp + fp > 0:
        precision = tp / (tp + fp)
    recall = 0.0
    if tp + fn > 0:
        recall = tp / (tp + fn)
    return precision, recall, f1_from_counts(tp, fp, fn)


def accuracy(true_labels, predictions):
    correct = 0
    for i in range(len(true_labels)):
        if true_labels[i] == predictions[i]:
            correct += 1
    return correct / len(true_labels)


# ---------- sentiment (one class per review) ----------

def confusion_matrix(true_labels, predictions, num_classes):
    # matrix[true][predicted] = how many reviews
    matrix = [[0] * num_classes for _ in range(num_classes)]
    for i in range(len(true_labels)):
        matrix[true_labels[i]][predictions[i]] += 1
    return matrix


def sentiment_class_scores(matrix):
    # Returns one (precision, recall, f1, support) tuple per class.
    num_classes = len(matrix)
    scores = []
    for c in range(num_classes):
        tp = matrix[c][c]
        fp = sum(matrix[r][c] for r in range(num_classes)) - tp   # predicted c, but was another class
        fn = sum(matrix[c]) - tp                                  # was c, but predicted another class
        precision, recall, f1 = precision_recall_f1(tp, fp, fn)
        scores.append((precision, recall, f1, sum(matrix[c])))
    return scores


def sentiment_scores(logits, labels):
    # logits: tensor (num_reviews, 3). Returns accuracy and macro-F1.
    predictions = torch.argmax(logits, dim=1).tolist()
    matrix = confusion_matrix(labels, predictions, len(SENTIMENT_LABELS))
    class_scores = sentiment_class_scores(matrix)
    macro_f1 = sum(score[2] for score in class_scores) / len(class_scores)
    return accuracy(labels, predictions), macro_f1


# ---------- aspects (21 yes/no decisions per review) ----------

def aspect_predictions(logits, threshold):
    # sigmoid turns a score into a probability; >= threshold means "aspect present".
    probabilities = torch.sigmoid(logits).numpy()
    return (probabilities >= threshold).astype(int)


def aspect_counts(true_matrix, predicted_matrix):
    # One dictionary of tp / fp / fn / tn per aspect. Both inputs are 0/1 arrays (reviews x aspects).
    counts = []
    for a in range(true_matrix.shape[1]):
        true_column = true_matrix[:, a]
        predicted_column = predicted_matrix[:, a]
        counts.append({
            "tp": int(((predicted_column == 1) & (true_column == 1)).sum()),
            "fp": int(((predicted_column == 1) & (true_column == 0)).sum()),
            "fn": int(((predicted_column == 0) & (true_column == 1)).sum()),
            "tn": int(((predicted_column == 0) & (true_column == 0)).sum()),
        })
    return counts


def aspect_scores(logits, labels, threshold):
    # Returns macro-F1 and micro-F1 over all aspects.
    predicted_matrix = aspect_predictions(logits, threshold)
    counts = aspect_counts(np.array(labels), predicted_matrix)

    f1_scores = []
    total_tp = total_fp = total_fn = 0
    for c in counts:
        f1_scores.append(f1_from_counts(c["tp"], c["fp"], c["fn"]))
        total_tp += c["tp"]
        total_fp += c["fp"]
        total_fn += c["fn"]

    macro_f1 = sum(f1_scores) / len(f1_scores)
    micro_f1 = f1_from_counts(total_tp, total_fp, total_fn)
    return macro_f1, micro_f1


def loss_on_logits(loss_function, logits, labels, label_dtype):
    # Validation loss, so we can compare it with the training loss.
    labels_tensor = torch.tensor(labels, dtype=label_dtype)
    return loss_function(logits, labels_tensor).item()

### Diagram helpers

Small functions that only draw. The numbers come from our own functions above.

In [ ]:
def plot_training_curves(train_loss, val_loss, score_curves, title, save_path):
    # Left: loss per epoch. Right: validation scores per epoch.
    epochs = list(range(1, len(train_loss) + 1))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(epochs, train_loss, marker="o", label="train loss")
    ax1.plot(epochs, val_loss, marker="o", label="validation loss")
    ax1.set_title(title + ": loss")
    ax1.set_xlabel("Epoch")
    ax1.set_xticks(epochs)
    ax1.legend()

    for name, values in score_curves.items():
        ax2.plot(epochs, values, marker="o", label=name)
    ax2.set_title(title + ": validation scores")
    ax2.set_xlabel("Epoch")
    ax2.set_xticks(epochs)
    ax2.set_ylim(0, 1)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()


def draw_matrix(ax, values, text_values, x_names, y_names, title):
    # Draw one heatmap; text_values are the numbers printed inside the cells.
    ax.imshow(values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(x_names)))
    ax.set_yticks(range(len(y_names)))
    ax.set_xticklabels(x_names)
    ax.set_yticklabels(y_names)
    ax.set_title(title)
    for r in range(len(y_names)):
        for c in range(len(x_names)):
            color = "white" if values[r][c] > 0.5 else "black"
            ax.text(c, r, text_values[r][c], ha="center", va="center", color=color)


def plot_confusion_matrix(matrix, class_names, title, save_path):
    # Left: raw counts. Right: each row divided by its total (= recall of that class).
    normalized = []
    for row in matrix:
        total = sum(row)
        if total == 0:
            normalized.append([0.0] * len(row))
        else:
            normalized.append([value / total for value in row])

    count_text = [[str(value) for value in row] for row in matrix]
    percent_text = [[f"{value * 100:.1f}%" for value in row] for row in normalized]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    draw_matrix(ax1, normalized, count_text, class_names, class_names, title + " (counts)")
    draw_matrix(ax2, normalized, percent_text, class_names, class_names, title + " (% of true class)")
    for ax in (ax1, ax2):
        ax.set_xlabel("Predicted label")
        ax.set_ylabel("True label")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

### Saving in the web app's format

The web app loads each model with `AutoModelForSequenceClassification.from_pretrained(folder)`,
reads the label names from the `id2label` entry in `config.json`, and reads
aspect thresholds from `thresholds.json`.

Our `BertClassifier` has the same pieces as `BertForSequenceClassification`
(`bert` encoder + `classifier` layer), so to save we create that standard model,
copy our trained weights into it, and call `save_pretrained`. We also save the
tokenizer in the same folder.

In [ ]:
def make_id2label(label_names):
    id2label = {}
    for i in range(len(label_names)):
        id2label[i] = label_names[i]
    return id2label


def save_for_web(model, folder, label_names, problem_type):
    standard_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=len(label_names),
        id2label=make_id2label(label_names),
        label2id={name: i for i, name in enumerate(label_names)},
        problem_type=problem_type,
    )
    # Copy our trained weights into the standard model.
    standard_model.bert.load_state_dict(model.bert.state_dict())
    standard_model.classifier.load_state_dict(model.classifier.state_dict())

    standard_model.save_pretrained(folder)
    tokenizer.save_pretrained(folder)


def load_model(folder):
    # Read a saved folder back into our own BertClassifier.
    standard_model = AutoModelForSequenceClassification.from_pretrained(folder)
    model = build_model(standard_model.config.num_labels)
    model.bert.load_state_dict(standard_model.bert.state_dict())
    model.classifier.load_state_dict(standard_model.classifier.state_dict())
    return model

## Step 7: Sentiment model (3 classes)

In [ ]:
train_sentiment_labels = [SENTIMENT_TO_ID[name] for name in train_df["sentiment"]]
validation_sentiment_labels = [SENTIMENT_TO_ID[name] for name in validation_df["sentiment"]]
test_sentiment_labels = [SENTIMENT_TO_ID[name] for name in test_df["sentiment"]]

sentiment_model = build_model(num_classes=len(SENTIMENT_LABELS))
sentiment_optimizer = torch.optim.AdamW(sentiment_model.parameters(), lr=LEARNING_RATE)
sentiment_loss_function = nn.CrossEntropyLoss()   # one correct class out of three

print(sentiment_model)

### Train the sentiment model

After every epoch we measure validation loss, accuracy and macro-F1, and keep only
the best checkpoint (by validation macro-F1). The best checkpoint is written to
`artifacts/sentiment` in the web app's format.

In [ ]:
best_sentiment_macro_f1 = -1.0
sentiment_history = {"train_loss": [], "val_loss": [], "val_accuracy": [], "val_macro_f1": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(sentiment_model, sentiment_optimizer, sentiment_loss_function,
                                 train_ids, train_sentiment_labels, torch.long)

    val_logits = get_logits(sentiment_model, validation_ids)
    val_loss = loss_on_logits(sentiment_loss_function, val_logits, validation_sentiment_labels, torch.long)
    val_accuracy, val_macro_f1 = sentiment_scores(val_logits, validation_sentiment_labels)

    sentiment_history["train_loss"].append(train_loss)
    sentiment_history["val_loss"].append(val_loss)
    sentiment_history["val_accuracy"].append(val_accuracy)
    sentiment_history["val_macro_f1"].append(val_macro_f1)

    print(f"Epoch {epoch}: train loss = {train_loss:.4f}, validation loss = {val_loss:.4f}, "
          f"validation accuracy = {val_accuracy:.4f}, validation macro-F1 = {val_macro_f1:.4f}")

    if val_macro_f1 > best_sentiment_macro_f1:
        best_sentiment_macro_f1 = val_macro_f1
        save_for_web(sentiment_model, SENTIMENT_DIR, SENTIMENT_LABELS, "single_label_classification")
        print("  New best sentiment model saved.")

In [ ]:
plot_training_curves(
    sentiment_history["train_loss"],
    sentiment_history["val_loss"],
    {"accuracy": sentiment_history["val_accuracy"], "macro-F1": sentiment_history["val_macro_f1"]},
    "Sentiment",
    PLOTS_DIR / "sentiment_training_curves.png",
)

### Test the best sentiment model

We reload the best checkpoint (not necessarily the last epoch) and score it once
on the test set, then draw the **confusion matrix**.

In [ ]:
sentiment_model = load_model(SENTIMENT_DIR)

test_logits = get_logits(sentiment_model, test_ids)
test_predictions = torch.argmax(test_logits, dim=1).tolist()

test_accuracy, test_macro_f1 = sentiment_scores(test_logits, test_sentiment_labels)
print(f"Sentiment test accuracy = {test_accuracy:.4f}, test macro-F1 = {test_macro_f1:.4f}\n")

sentiment_matrix = confusion_matrix(test_sentiment_labels, test_predictions, len(SENTIMENT_LABELS))
class_scores = sentiment_class_scores(sentiment_matrix)

print(f"{'class':<10} {'precision':>10} {'recall':>8} {'f1':>8} {'support':>8}")
for i in range(len(SENTIMENT_LABELS)):
    precision, recall, f1, support = class_scores[i]
    print(f"{SENTIMENT_LABELS[i]:<10} {precision:>10.4f} {recall:>8.4f} {f1:>8.4f} {support:>8}")

plot_confusion_matrix(sentiment_matrix, SENTIMENT_LABELS, "Sentiment confusion matrix",
                      PLOTS_DIR / "sentiment_confusion_matrix.png")

In [ ]:
# Bar chart: precision, recall and F1 for each sentiment class.
positions = np.arange(len(SENTIMENT_LABELS))
width = 0.25

plt.figure(figsize=(7, 4))
plt.bar(positions - width, [s[0] for s in class_scores], width, label="precision")
plt.bar(positions, [s[1] for s in class_scores], width, label="recall")
plt.bar(positions + width, [s[2] for s in class_scores], width, label="F1")
plt.xticks(positions, SENTIMENT_LABELS)
plt.ylim(0, 1)
plt.title("Sentiment: per-class scores on the test set")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "sentiment_class_scores.png", dpi=150)
plt.show()

sentiment_metrics = {
    "accuracy": test_accuracy,
    "macro_f1": test_macro_f1,
    "per_class": {SENTIMENT_LABELS[i]: {"precision": class_scores[i][0], "recall": class_scores[i][1],
                                        "f1": class_scores[i][2], "support": class_scores[i][3]}
                  for i in range(len(SENTIMENT_LABELS))},
    "confusion_matrix": sentiment_matrix,
}
with open(SENTIMENT_DIR / "metrics.json", "w") as f:
    json.dump(sentiment_metrics, f, indent=2)

## Step 8: Aspect model (21 yes/no labels)

A review can mention several aspects at once, so each aspect is its own
yes/no question. We use `BCEWithLogitsLoss` (binary cross-entropy on every
aspect) instead of cross-entropy, and a sigmoid + threshold to decide "yes".

In [ ]:
train_aspect_labels = train_df[ASPECT_COLUMNS].values.tolist()
validation_aspect_labels = validation_df[ASPECT_COLUMNS].values.tolist()
test_aspect_labels = test_df[ASPECT_COLUMNS].values.tolist()

aspect_model = build_model(num_classes=len(ASPECT_COLUMNS))
aspect_optimizer = torch.optim.AdamW(aspect_model.parameters(), lr=LEARNING_RATE)
aspect_loss_function = nn.BCEWithLogitsLoss()

### Train the aspect model

While training we judge each epoch with the default threshold 0.5. The best
checkpoint (by validation macro-F1) is written to `artifacts/aspects`.

In [ ]:
best_aspect_macro_f1 = -1.0
aspect_history = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "val_micro_f1": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(aspect_model, aspect_optimizer, aspect_loss_function,
                                 train_ids, train_aspect_labels, torch.float32)

    val_logits = get_logits(aspect_model, validation_ids)
    val_loss = loss_on_logits(aspect_loss_function, val_logits, validation_aspect_labels, torch.float32)
    val_macro_f1, val_micro_f1 = aspect_scores(val_logits, validation_aspect_labels, threshold=0.5)

    aspect_history["train_loss"].append(train_loss)
    aspect_history["val_loss"].append(val_loss)
    aspect_history["val_macro_f1"].append(val_macro_f1)
    aspect_history["val_micro_f1"].append(val_micro_f1)

    print(f"Epoch {epoch}: train loss = {train_loss:.4f}, validation loss = {val_loss:.4f}, "
          f"validation macro-F1 = {val_macro_f1:.4f}, validation micro-F1 = {val_micro_f1:.4f}")

    if val_macro_f1 > best_aspect_macro_f1:
        best_aspect_macro_f1 = val_macro_f1
        save_for_web(aspect_model, ASPECT_DIR, ASPECT_COLUMNS, "multi_label_classification")
        print("  New best aspect model saved.")

In [ ]:
plot_training_curves(
    aspect_history["train_loss"],
    aspect_history["val_loss"],
    {"macro-F1": aspect_history["val_macro_f1"], "micro-F1": aspect_history["val_micro_f1"]},
    "Aspects",
    PLOTS_DIR / "aspect_training_curves.png",
)

In [ ]:
# Reload the best checkpoint and score it on the test set with the default threshold 0.5.
aspect_model = load_model(ASPECT_DIR)

validation_logits = get_logits(aspect_model, validation_ids)   # reused for the threshold search
test_aspect_logits = get_logits(aspect_model, test_ids)

test_macro_f1, test_micro_f1 = aspect_scores(test_aspect_logits, test_aspect_labels, threshold=0.5)
print(f"Aspect test macro-F1 = {test_macro_f1:.4f}, test micro-F1 = {test_micro_f1:.4f}  (threshold 0.5)")

## Step 9: Pick a better aspect threshold

So far we called a probability "yes" whenever it was at least 0.5. Here we
try a small range of thresholds on the validation set and keep whichever one
gives the best macro-F1. The curve shows how the score changes with the threshold.

In [ ]:
best_threshold = 0.5
best_threshold_macro_f1 = -1.0

sweep_thresholds = []
sweep_macro_f1 = []
sweep_micro_f1 = []

threshold = 0.20
while threshold <= 0.80:
    threshold = round(threshold, 2)
    macro_f1, micro_f1 = aspect_scores(validation_logits, validation_aspect_labels, threshold)
    print(f"threshold = {threshold:.2f}, validation macro-F1 = {macro_f1:.4f}")

    sweep_thresholds.append(threshold)
    sweep_macro_f1.append(macro_f1)
    sweep_micro_f1.append(micro_f1)

    if macro_f1 > best_threshold_macro_f1:
        best_threshold_macro_f1 = macro_f1
        best_threshold = threshold

    threshold += 0.05

print("Best threshold:", best_threshold)

plt.figure(figsize=(7, 4))
plt.plot(sweep_thresholds, sweep_macro_f1, marker="o", label="macro-F1")
plt.plot(sweep_thresholds, sweep_micro_f1, marker="o", label="micro-F1")
plt.axvline(best_threshold, color="red", linestyle="--", label=f"best = {best_threshold}")
plt.title("Aspect threshold search (validation set)")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "aspect_threshold_search.png", dpi=150)
plt.show()

In [ ]:
# The web app reads "per_label_thresholds" (one value per aspect) from thresholds.json.
# We use the same best threshold for every aspect.
per_label_thresholds = {}
for aspect in ASPECT_COLUMNS:
    per_label_thresholds[aspect] = best_threshold

threshold_info = {"global_threshold": best_threshold, "per_label_thresholds": per_label_thresholds}
with open(ASPECT_DIR / "thresholds.json", "w") as f:
    json.dump(threshold_info, f, indent=2)

## Step 10: Final aspect results on the test set

Now we score the aspect model on the test set with the chosen threshold and draw:

1. Precision / recall / F1 for every aspect.
2. One **confusion matrix per aspect**. Each is a 2x2 table: rows = truth
   (no / yes), columns = prediction (no / yes).

In [ ]:
test_macro_f1, test_micro_f1 = aspect_scores(test_aspect_logits, test_aspect_labels, best_threshold)
print(f"Aspect test macro-F1 = {test_macro_f1:.4f}, test micro-F1 = {test_micro_f1:.4f}  (threshold {best_threshold})\n")

test_predicted_matrix = aspect_predictions(test_aspect_logits, best_threshold)
test_counts = aspect_counts(np.array(test_aspect_labels), test_predicted_matrix)

aspect_rows = []
print(f"{'aspect':<26} {'precision':>9} {'recall':>7} {'f1':>7} {'support':>8}")
for a in range(len(ASPECT_COLUMNS)):
    c = test_counts[a]
    precision, recall, f1 = precision_recall_f1(c["tp"], c["fp"], c["fn"])
    support = c["tp"] + c["fn"]
    aspect_rows.append((ASPECT_COLUMNS[a], precision, recall, f1, support))
    print(f"{ASPECT_COLUMNS[a]:<26} {precision:>9.4f} {recall:>7.4f} {f1:>7.4f} {support:>8}")

In [ ]:
# Bar chart: precision, recall and F1 for every aspect, best F1 at the top.
sorted_rows = sorted(aspect_rows, key=lambda row: row[3])
names = [f"{row[0]} (n={row[4]})" for row in sorted_rows]
positions = np.arange(len(sorted_rows))
height = 0.27

plt.figure(figsize=(9, 9))
plt.barh(positions + height, [row[1] for row in sorted_rows], height, label="precision")
plt.barh(positions, [row[2] for row in sorted_rows], height, label="recall")
plt.barh(positions - height, [row[3] for row in sorted_rows], height, label="F1")
plt.yticks(positions, names)
plt.xlim(0, 1)
plt.title(f"Aspects: per-aspect scores on the test set (threshold {best_threshold})")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "aspect_class_scores.png", dpi=150)
plt.show()

In [ ]:
# One 2x2 confusion matrix per aspect (5 x 5 grid, the last 4 boxes stay empty).
fig, axes = plt.subplots(5, 5, figsize=(16, 16))
axes = axes.flatten()

for a in range(len(ASPECT_COLUMNS)):
    c = test_counts[a]
    matrix = [[c["tn"], c["fp"]],
              [c["fn"], c["tp"]]]

    # Colour by row percentage so rare aspects are still readable.
    shades = []
    for row in matrix:
        total = sum(row)
        shades.append([value / total if total > 0 else 0.0 for value in row])
    text = [[str(value) for value in row] for row in matrix]

    draw_matrix(axes[a], shades, text, ["no", "yes"], ["no", "yes"], ASPECT_COLUMNS[a])
    axes[a].set_xlabel("Predicted")
    axes[a].set_ylabel("True")

for a in range(len(ASPECT_COLUMNS), len(axes)):
    axes[a].axis("off")

plt.suptitle("Confusion matrix for each aspect (test set)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(PLOTS_DIR / "aspect_confusion_matrices.png", dpi=150)
plt.show()

aspect_metrics = {
    "threshold": best_threshold,
    "macro_f1": test_macro_f1,
    "micro_f1": test_micro_f1,
    "per_aspect": {row[0]: {"precision": row[1], "recall": row[2], "f1": row[3], "support": row[4]}
                   for row in aspect_rows},
}
with open(ASPECT_DIR / "metrics.json", "w") as f:
    json.dump(aspect_metrics, f, indent=2)

## Step 11: Try the models out

Both models are still in memory, so we can use them right away.

In [ ]:
def predict_sentiment(text):
    ids = encode_all([text])[0]
    logits = get_logits(sentiment_model, [ids])[0]

    probabilities = torch.softmax(logits, dim=0).tolist()
    scores = {}
    for i in range(len(SENTIMENT_LABELS)):
        scores[SENTIMENT_LABELS[i]] = round(probabilities[i], 4)

    predicted_id = int(torch.argmax(logits))
    predicted_label = SENTIMENT_LABELS[predicted_id]

    return predicted_label, scores


def predict_aspects(text):
    ids = encode_all([text])[0]
    logits = get_logits(aspect_model, [ids])[0]

    probabilities = torch.sigmoid(logits).tolist()

    detected_aspects = []
    for i in range(len(ASPECT_COLUMNS)):
        if probabilities[i] >= best_threshold:
            detected_aspects.append((ASPECT_COLUMNS[i], round(probabilities[i], 4)))

    detected_aspects.sort(key=lambda item: item[1], reverse=True)
    return detected_aspects

In [ ]:
example_sentences = [
    "The staff were friendly and the room was spotless.",
    "The Wi-Fi was slow and the room was noisy.",
    "Breakfast was poor but the staff were excellent.",
    "The hotel is located three kilometres from the airport.",
]

for sentence in example_sentences:
    predicted_label, scores = predict_sentiment(sentence)
    aspects_found = predict_aspects(sentence)

    print("Review:", sentence)
    print("Sentiment:", predicted_label, scores)
    print("Aspects:", aspects_found)
    print()

## Step 12: Check the saved files and pack them for the web app

First we load the saved folders exactly the way `src/inference/predictor.py`
does (`AutoTokenizer` + `AutoModelForSequenceClassification` + label names from
`config.json` + `thresholds.json`) and predict one sentence. If this works, the
web app can use the same folders.

In [ ]:
web_text = "Breakfast was poor but the staff were excellent."

# ----- sentiment, loaded like the web app does -----
web_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_DIR)
web_sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_DIR)
web_sentiment_model.eval()

encoded = web_tokenizer(web_text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
with torch.no_grad():
    web_logits = web_sentiment_model(**encoded).logits
web_probabilities = torch.softmax(web_logits, dim=1)[0].tolist()

print("Sentiment label names in config.json:", web_sentiment_model.config.id2label)
for i in range(len(web_probabilities)):
    print(f"  {web_sentiment_model.config.id2label[i]}: {web_probabilities[i]:.4f}")

# The in-memory model should give (almost) the same numbers as the reloaded one.
memory_logits = get_logits(sentiment_model, encode_all([web_text]))
print("Max difference vs in-memory model:", float((memory_logits - web_logits).abs().max()))

# ----- aspects, loaded like the web app does -----
web_aspect_model = AutoModelForSequenceClassification.from_pretrained(ASPECT_DIR)
web_aspect_model.eval()
with open(ASPECT_DIR / "thresholds.json") as f:
    web_thresholds = json.load(f)["per_label_thresholds"]

with torch.no_grad():
    aspect_logits_web = web_aspect_model(**encoded).logits
aspect_probabilities = torch.sigmoid(aspect_logits_web)[0].tolist()

print("\nDetected aspects:")
for i in range(len(aspect_probabilities)):
    name = web_aspect_model.config.id2label[i]
    if aspect_probabilities[i] >= web_thresholds[name]:
        print(f"  {name}: {aspect_probabilities[i]:.4f}")

In [ ]:
# Pack both model folders into one zip so it is easy to download from the Output tab.
zip_path = shutil.make_archive(str(WORKING_DIR / "hotel_models"), "zip",
                               root_dir=WORKING_DIR, base_dir="artifacts")
print("Created:", zip_path)

print("\nSaved files:")
for folder in (SENTIMENT_DIR, ASPECT_DIR):
    for file in sorted(folder.iterdir()):
        print(f"  {file.relative_to(WORKING_DIR)}  ({file.stat().st_size / 1e6:.1f} MB)")

print("\nDiagrams:")
for file in sorted(PLOTS_DIR.iterdir()):
    print("  ", file.relative_to(WORKING_DIR))

### Use the models in the web app

1. Click **Save Version -> Save & Run All (Commit)** so Kaggle keeps the output.
2. Open the saved version's **Output** tab and download `hotel_models.zip`.
3. Unzip it in the project root, so these folders exist:
   `artifacts/sentiment` and `artifacts/aspects`.
4. Run `python src/web/app.py`. (The app looks in those two folders by default.)